# 03 — Tree DFS and BFS

## Why This Matters for DSA
Tree traversals (`02_Trees_and_BST`) are just the starting point. Most tree problems require you to perform complex computations during the walk. We solve these problems by framing them as **Depth-First Search (DFS)**—diving down branches recursively—or **Breadth-First Search (BFS)**—exploring level-by-level. Tree DFS and BFS are the foundation for general graph algorithms, backtracking, and dynamic programming on trees. Mastering the flow of recursion (top-down vs. bottom-up), managing recursive state (tree backtracking), and level size tracking in BFS is what transforms basic traversal into a powerful tool for solving complex hierarchical problems.

## Prerequisites
- `02_Recursion` (Phase 01) — complete comfort with recursive execution and call stacks
- `08_Stacks_Queues_Deques` (Phase 02) — understanding queue (FIFO) and stack (LIFO) mechanics
- `02_Trees_and_BST` (Phase 03) — tree terminology, dynamic linked node representation (`TreeNode`), and basic preorder/inorder/postorder traversals

## Learning Objectives
By the end of this notebook, you will be able to:
- Distinguish between **Top-Down DFS** (passing state down) and **Bottom-Up DFS** (bubbling results up) and choose the correct paradigm for any tree problem.
- Implement tree backtracking algorithms to find root-to-leaf paths while managing shared state with $O(h)$ space complexity.
- Implement level-order tree BFS with precise **level size tracking** to process nodes level-by-level.
- Convert recursive tree traversals into iterative implementations using an explicit `std::stack`.
- Solve the **Lowest Common Ancestor (LCA)** problem in a general binary tree in $O(n)$ time using postorder bubbling.
- Serialize and deserialize a binary tree into a string representation using preorder DFS.
- Compare the space-time bounds of DFS and BFS, explaining how tree shape (balanced vs. skewed) determines their memory footprint.

## Checklist Before Moving On
- [ ] I can explain the difference between top-down and bottom-up DFS in one sentence
- [ ] I can write a path backtracking function without allocating a new path vector at each recursive step
- [ ] I can write a level-order BFS that computes level-by-level statistics (e.g., right side view) using level size tracking
- [ ] I can implement iterative preorder and inorder traversals using an explicit stack
- [ ] I can explain the recursive pointer bubbling strategy for LCA in a general binary tree
- [ ] I can write tree serialization and deserialization functions from memory
- [ ] I can identify whether DFS or BFS uses more memory for a balanced tree vs. a skewed tree

## Table of Contents
1. Tree DFS Fundamentals — Top-Down vs. Bottom-Up Patterns
2. Tree Backtracking — Finding Root-to-Leaf Paths
3. Tree BFS Fundamentals — Level Size Tracking and Zigzag Traversal
4. Iterative DFS Traversals — Simulating Recursion with an Explicit Stack
5. Lowest Common Ancestor (LCA) in a General Binary Tree
6. Tree Serialization and Deserialization
7. Time and Space Complexity Analysis — Stack vs. Queue Bounds
8. Decision Guide — DFS vs. BFS for Tree Problems
9. Phase Checkpoint, Cheat Sheet, and Answer Key
10. LeetCode Practice Problems


## 1. Tree DFS Fundamentals — Top-Down vs. Bottom-Up Patterns

### The Why
Tree Depth-First Search (DFS) uses recursion to explore branches. The key to writing correct and optimal DFS algorithms is organizing the flow of information. We divide DFS into two primary patterns: **Top-Down** (passing values down the recursive calls) and **Bottom-Up** (gathering values from subtrees and returning them up). Choosing the wrong flow leads to redundant computations ($O(n^2)$ time) or overly complex state tracking.

### Core Mechanism
- **Top-Down DFS (Preorder Flow):**
  - Information is passed from the parent to child nodes via function arguments.
  - Calculations and actions happen on the way *down* before traversing children.
  - **Example: Path Sum.** To check if a root-to-leaf path sums to `targetSum`, we subtract the parent node's value from `targetSum` and pass the remaining sum to its children. The base case checks if a leaf node's value equals the remaining sum.
- **Bottom-Up DFS (Postorder Flow / Divide & Conquer):**
  - Information is gathered from the child subtrees *first*, combined at the current node, and returned up to the parent.
  - Calculations happen on the way *up* (postorder visit).
  - **Example: Binary Tree Diameter.** The diameter of a tree is the maximum path length between any two nodes. The maximum path passing through node $N$ is (height of left subtree + 1) + (height of right subtree + 1). The recursive function computes height bottom-up and updates the maximum diameter found so far.

### Common Pitfalls
- **Redundant Re-evaluations:** Trying to solve bottom-up problems top-down. For example, computing tree height at every node to check if a tree is balanced. This results in $O(n^2)$ time because height is recalculated repeatedly. Doing it bottom-up solves it in $O(n)$ time by returning height and balance status in a single pass.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <algorithm>
using namespace std;

struct TreeNode {
    int value;
    TreeNode *left, *right;
    TreeNode(int val) : value(val), left(nullptr), right(nullptr) {}
};

// 1. TOP-DOWN DFS: Path Sum
// We pass the remaining sum DOWN to the children. The work is done on the way down.
bool hasPathSum(TreeNode* root, int targetSum) {
    if (root == nullptr) return false;
    
    // Base case: leaf node
    if (root->left == nullptr && root->right == nullptr) {
        return targetSum == root->value;
    }
    
    int remainingSum = targetSum - root->value;
    return hasPathSum(root->left, remainingSum) || 
           hasPathSum(root->right, remainingSum);
}

// 2. BOTTOM-UP DFS: Binary Tree Diameter
// We gather subtree heights from the bottom up, update diameter, and return height.
int calculateHeightAndDiameter(TreeNode* root, int& diameter) {
    if (root == nullptr) return -1; // Height of null is -1
    
    int leftHeight = calculateHeightAndDiameter(root->left, diameter);
    int rightHeight = calculateHeightAndDiameter(root->right, diameter);
    
    // The longest path passing through root is (leftHeight + 1) + (rightHeight + 1)
    int currentPathLength = (leftHeight + 1) + (rightHeight + 1);
    diameter = max(diameter, currentPathLength);
    
    return 1 + max(leftHeight, rightHeight); // Return height up to parent
}

int getDiameter(TreeNode* root) {
    int diameter = 0;
    calculateHeightAndDiameter(root, diameter);
    return diameter;
}

void cleanTree(TreeNode *root) {
    if (root) {
        cleanTree(root->left);
        cleanTree(root->right);
        delete root;
    }
}

int main() {
    // Construct tree:
    //         1
    //       /   \
    //      2     3
    //     / \     \
    //    4   5     6
    TreeNode* root = new TreeNode(1);
    root->left = new TreeNode(2);
    root->right = new TreeNode(3);
    root->left->left = new TreeNode(4);
    root->left->right = new TreeNode(5);
    root->right->right = new TreeNode(6);

    // Top-down check
    cout << "Path Sum 7 (1->2->4)? " << (hasPathSum(root, 7) ? "Yes" : "No") << "\n";
    
    // Bottom-up check
    cout << "Binary Tree Diameter:   " << getDiameter(root) << "\n"; // Expected: 4 (4->2->1->3->6)

    cleanTree(root);
    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 2. Tree Backtracking — Finding Root-to-Leaf Paths

### The Why
Tree Backtracking is a specific top-down DFS pattern. It is used when we need to find root-to-leaf paths that satisfy a condition. To do this efficiently, we maintain a single, shared path list. We push elements onto this list as we walk down, and pop them (backtrack) as we go back up. This avoids copying vectors at every node, keeping space complexity bounded by $O(h)$ instead of $O(n^2)$.

### Core Mechanism
- **Shared State Pattern:**
  - Pass the current path vector **by reference** (`vector<int>& currentPath`).
  - In the recursive step:
    1. Push the current node's value onto `currentPath`.
    2. If the node is a leaf and meets the criteria, copy `currentPath` to the global results.
    3. Recurse on the left and right subtrees.
    4. **Backtrack:** Pop the current node's value from `currentPath` before the function returns.
- **Complexity:** $O(n)$ time to visit each node once, $O(h)$ auxiliary space for the recursion stack and path tracking.

### Common Pitfalls
- **Passing Vectors by Value:** Declaring the recursive helper as `void helper(TreeNode* root, vector<int> path)`. This forces a copy of the vector at every single node, consuming $O(n h)$ memory and slowing down execution. Always pass the path by reference (`vector<int>& path`) and backtrack manually.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

struct TreeNode {
    int value;
    TreeNode *left, *right;
    TreeNode(int val) : value(val), left(nullptr), right(nullptr) {}
};

// Recursive helper utilizing a single shared path vector by reference
void pathSumHelper(TreeNode* root, int remainingSum, vector<int>& currentPath, vector<vector<int>>& results) {
    if (root == nullptr) return;
    
    // 1. Choose: add current node value to path
    currentPath.push_back(root->value);
    
    // 2. Check: is leaf node and sum matches?
    if (root->left == nullptr && root->right == nullptr) {
        if (remainingSum == root->value) {
            results.push_back(currentPath); // Save path copy
        }
    }
    
    // 3. Recurse
    pathSumHelper(root->left, remainingSum - root->value, currentPath, results);
    pathSumHelper(root->right, remainingSum - root->value, currentPath, results);
    
    // 4. Backtrack: remove current node value before returning up
    currentPath.pop_back();
}

vector<vector<int>> pathSum(TreeNode* root, int targetSum) {
    vector<vector<int>> results;
    vector<int> currentPath;
    pathSumHelper(root, targetSum, currentPath, results);
    return results;
}

void cleanTree(TreeNode *root) {
    if (root) {
        cleanTree(root->left);
        cleanTree(root->right);
        delete root;
    }
}

int main() {
    // Construct tree:
    //         5
    //       /   \
    //      4     8
    //     /     / \
    //    11    13  4
    //   /  \      / \
    //  7    2    5   1
    TreeNode* root = new TreeNode(5);
    root->left = new TreeNode(4);
    root->right = new TreeNode(8);
    root->left->left = new TreeNode(11);
    root->left->left->left = new TreeNode(7);
    root->left->left->right = new TreeNode(2);
    root->right->left = new TreeNode(13);
    root->right->right = new TreeNode(4);
    root->right->right->left = new TreeNode(5);
    root->right->right->right = new TreeNode(1);

    vector<vector<vector<int>>> test_cases; // dummy vector
    vector<vector<int>> paths = pathSum(root, 22);
    
    cout << "Paths summing to 22:\n";
    for (const auto& path : paths) {
        for (int val : path) {
            cout << val << " ";
        }
        cout << "\n";
    }
    // Expected output:
    // 5 4 11 2 
    // 5 8 4 5 

    cleanTree(root);
    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 3. Tree BFS Fundamentals — Level Size Tracking and Zigzag Traversal

### The Why
Tree Breadth-First Search (BFS) is level-order traversal. Standard BFS prints nodes in FIFO order. However, advanced level-order problems require grouping nodes *by level* (e.g., finding the average of each level, finding the rightmost visible node, or printing levels in a zigzag pattern). To do this, we must use the **Level Size Tracking** pattern, which isolates level boundaries.

### Core Mechanism
- **Level Size Tracking Pattern:**
  1. Enqueue the root. Outer loop runs while the queue is not empty.
  2. At the start of each level, record `int levelSize = q.size()`. This is exactly the number of nodes at the current depth.
  3. Run an inner loop exactly `levelSize` times. Process these nodes and enqueue their children. Any nodes pushed during this step belong to the *next* level and will not be processed until the next outer loop iteration.
- **Zigzag Traversal:** Alternate the insertion direction of each level. For even levels, fill the level's result array from left to right; for odd levels, fill it from right to left.
- **Right Side View:** The rightmost node at any level is simply the last node processed in the inner level loop (`i == levelSize - 1`).

### Common Pitfalls
- **Using Dynamic Queue Size:** Writing `for (int i = 0; i < q.size(); i++)`. The queue size changes as children are enqueued, which will cause the loop to run indefinitely or mix different levels together. Always save the size into `levelSize` *before* the inner loop begins.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <queue>
#include <algorithm>
using namespace std;

struct TreeNode {
    int value;
    TreeNode *left, *right;
    TreeNode(int val) : value(val), left(nullptr), right(nullptr) {}
};

// 1. BFS Right Side View using Level Size Tracking
vector<int> rightSideView(TreeNode* root) {
    vector<int> result;
    if (root == nullptr) return result;
    
    queue<TreeNode*> q;
    q.push(root);
    
    while (!q.empty()) {
        int levelSize = q.size(); // CAPTURE LEVEL SIZE FIRST
        
        for (int i = 0; i < levelSize; i++) {
            TreeNode* cur = q.front();
            q.pop();
            
            // The last node in the inner loop is the rightmost node of the level
            if (i == levelSize - 1) {
                result.push_back(cur->value);
            }
            
            if (cur->left != nullptr) q.push(cur->left);
            if (cur->right != nullptr) q.push(cur->right);
        }
    }
    return result;
}

// 2. BFS Zigzag Level Order Traversal
vector<vector<int>> zigzagLevelOrder(TreeNode* root) {
    vector<vector<int>> result;
    if (root == nullptr) return result;
    
    queue<TreeNode*> q;
    q.push(root);
    bool leftToRight = true;
    
    while (!q.empty()) {
        int levelSize = q.size();
        vector<int> currentLevel(levelSize);
        
        for (int i = 0; i < levelSize; i++) {
            TreeNode* cur = q.front();
            q.pop();
            
            // Insert in normal or reverse index order based on flag
            int index = leftToRight ? i : (levelSize - 1 - i);
            currentLevel[index] = cur->value;
            
            if (cur->left != nullptr) q.push(cur->left);
            if (cur->right != nullptr) q.push(cur->right);
        }
        result.push_back(currentLevel);
        leftToRight = !leftToRight; // Flip direction for the next level
    }
    return result;
}

void cleanTree(TreeNode *root) {
    if (root) {
        cleanTree(root->left);
        cleanTree(root->right);
        delete root;
    }
}

int main() {
    // Construct tree:
    //         3
    //       /   \
    //      9     20
    //           /  \
    //          15   7
    TreeNode* root = new TreeNode(3);
    root->left = new TreeNode(9);
    root->right = new TreeNode(20);
    root->right->left = new TreeNode(15);
    root->right->right = new TreeNode(7);

    vector<int> rsv = rightSideView(root);
    cout << "Right Side View: ";
    for (int x : rsv) cout << x << " ";
    cout << "\n\n"; // Expected: 3 20 7

    vector<vector<int>> zigzag = zigzagLevelOrder(root);
    cout << "Zigzag Level Order:\n";
    for (const auto& lvl : zigzag) {
        for (int x : lvl) cout << x << " ";
        cout << "\n";
    }
    // Expected output:
    // 3 
    // 20 9 
    // 15 7 

    cleanTree(root);
    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 4. Iterative DFS Traversals — Simulating Recursion with an Explicit Stack

### The Why
Every recursive function utilizes the OS call stack. For deep trees, recursion risks stack overflow. In production systems, we avoid this by rewriting traversals iteratively using an explicit `std::stack` allocated on the heap (which has far larger space bounds). Simulating recursion helps you understand exactly how the compiler manages call frames.

### Core Mechanism
- **Iterative Preorder (V-L-R):**
  - Push the root onto the stack.
  - While the stack is not empty:
    1. Pop node `cur` and visit it.
    2. Push `cur->right` (if non-null).
    3. Push `cur->left` (if non-null).
  - *Why Right before Left:* Stack is LIFO, so pushing the right child first guarantees the left child is popped and visited first.
- **Iterative Inorder (L-V-R):**
  - Maintain current pointer `cur = root` and an empty stack.
  - While `cur != nullptr` or the stack is not empty:
    1. Inner loop: Push `cur` and move to `cur->left` until `cur` is null (going deep left).
    2. Pop the top node from stack and visit it.
    3. Set `cur = popped->right` to traverse its right subtree.

### Common Pitfalls
- **Pushing Children in the Wrong Order:** In iterative preorder, pushing the left child first. This will cause the right child to be popped and processed first, resulting in V-R-L order instead of V-L-R.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <stack>
using namespace std;

struct TreeNode {
    int value;
    TreeNode *left, *right;
    TreeNode(int val) : value(val), left(nullptr), right(nullptr) {}
};

// Iterative Preorder: Visit, Left, Right
void iterativePreorder(TreeNode* root) {
    if (root == nullptr) return;
    stack<TreeNode*> s;
    s.push(root);
    
    while (!s.empty()) {
        TreeNode* cur = s.top();
        s.pop();
        cout << cur->value << " ";
        
        // Push right first so left is popped and visited first (LIFO)
        if (cur->right != nullptr) s.push(cur->right);
        if (cur->left != nullptr) s.push(cur->left);
    }
    cout << "\n";
}

// Iterative Inorder: Left, Visit, Right
void iterativeInorder(TreeNode* root) {
    stack<TreeNode*> s;
    TreeNode* cur = root;
    
    while (cur != nullptr || !s.empty()) {
        // Go deep left, pushing all nodes to stack
        while (cur != nullptr) {
            s.push(cur);
            cur = cur->left;
        }
        // Pop and visit
        cur = s.top();
        s.pop();
        cout << cur->value << " ";
        
        // Transition to right child
        cur = cur->right;
    }
    cout << "\n";
}

void cleanTree(TreeNode *root) {
    if (root) {
        cleanTree(root->left);
        cleanTree(root->right);
        delete root;
    }
}

int main() {
    // Construct tree:
    //         1
    //       /   \
    //      2     3
    //     / \   /
    //    4   5 6
    TreeNode* root = new TreeNode(1);
    root->left = new TreeNode(2);
    root->right = new TreeNode(3);
    root->left->left = new TreeNode(4);
    root->left->right = new TreeNode(5);
    root->right->left = new TreeNode(6);

    cout << "Iterative Preorder: ";
    iterativePreorder(root); // Expected: 1 2 4 5 3 6

    cout << "Iterative Inorder:  ";
    iterativeInorder(root);  // Expected: 4 2 5 1 6 3

    cleanTree(root);
    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 5. Lowest Common Ancestor (LCA) in a General Binary Tree

### The Why
Finding the Lowest Common Ancestor (LCA) in a general binary tree is more challenging than in a BST (`02_Trees_and_BST` Section 9). Because the nodes are not ordered numerically, we cannot use quick BST key comparisons. Instead, we must search the entire tree using a bottom-up postorder DFS, bubbling up match pointers.

### Core Mechanism
- **Recursive Pointer Bubbling Strategy:**
  - Define `TreeNode* lowestCommonAncestor(TreeNode* root, TreeNode* p, TreeNode* q)`.
  - **Base Case:** If current node is null, or matches `p`, or matches `q`, return it directly.
  - **Divide:** Search left and right subtrees recursively: 
    - `leftLCA = lowestCommonAncestor(root->left, p, q)`
    - `rightLCA = lowestCommonAncestor(root->right, p, q)`
  - **Conquer & Bubble Up:**
    1. If *both* `leftLCA` and `rightLCA` are non-null: We found the split point. One target is in the left subtree, and the other is in the right. The current node is the LCA. Return current node.
    2. If only one is non-null: Return the non-null pointer (both targets reside in that subtree, or one is the ancestor of the other).
    3. If both are null: Return `nullptr`.
- **Complexity:** $O(n)$ time to visit each node once, $O(h)$ space for the call stack.

### Common Pitfalls
- **Incorrectly Assuming BST Ordering:** Trying to perform range checks like `if (p->val < root->val)`. In a general binary tree, nodes are placed arbitrarily, so we must inspect all paths.


In [ ]:
%%writefile temp.cpp
#include <iostream>
using namespace std;

struct TreeNode {
    int value;
    TreeNode *left, *right;
    TreeNode(int val) : value(val), left(nullptr), right(nullptr) {}
};

// Lowest Common Ancestor (LCA) in a general Binary Tree
TreeNode* lowestCommonAncestor(TreeNode* root, TreeNode* p, TreeNode* q) {
    // Base Case: null node, or found either target node
    if (root == nullptr || root == p || root == q) return root;
    
    // Divide: search both subtrees
    TreeNode* leftLCA = lowestCommonAncestor(root->left, p, q);
    TreeNode* rightLCA = lowestCommonAncestor(root->right, p, q);
    
    // Conquer: merge results bottom-up
    if (leftLCA != nullptr && rightLCA != nullptr) {
        return root; // Split point found: current node is the LCA
    }
    
    // If only one side is non-null, bubble that target up
    return (leftLCA != nullptr) ? leftLCA : rightLCA;
}

void cleanTree(TreeNode *root) {
    if (root) {
        cleanTree(root->left);
        cleanTree(root->right);
        delete root;
    }
}

int main() {
    // Construct tree:
    //         3
    //       /   \
    //      5     1
    //     / \   / \
    //    6   2 0   8
    //       / \
    //      7   4
    TreeNode* root = new TreeNode(3);
    root->left = new TreeNode(5);
    root->right = new TreeNode(1);
    root->left->left = new TreeNode(6);
    root->left->right = new TreeNode(2);
    root->left->right->left = new TreeNode(7);
    root->left->right->right = new TreeNode(4);
    root->right->left = new TreeNode(0);
    root->right->right = new TreeNode(8);

    TreeNode* p = root->left; // Node 5
    TreeNode* q = root->left->right->right; // Node 4
    
    TreeNode* lca = lowestCommonAncestor(root, p, q);
    cout << "LCA of 5 and 4: " << (lca ? lca->value : -1) << "\n"; // Expected: 5 (5 is parent of 4)

    TreeNode* r = root->left->left; // Node 6
    TreeNode* lca2 = lowestCommonAncestor(root, r, q);
    cout << "LCA of 6 and 4: " << (lca2 ? lca2->value : -1) << "\n"; // Expected: 5

    cleanTree(root);
    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 6. Tree Serialization and Deserialization

### The Why
Binary trees are dynamic structures stored in non-contiguous heap memory via pointers. To store a tree in a file, send it over a network, or display it as a text string, we must **serialize** it into a flat, linear format. We must also be able to **deserialize** it to reconstruct the exact tree structure in memory.

### Core Mechanism
- **Preorder DFS Approach:**
  - **Serialization:** Perform a preorder DFS traversal. Output each node value followed by a delimiter (e.g., `","`). When hitting a null pointer, output a sentinel character (e.g., `"#"`).
  - **Deserialization:** Read the tokens sequentially. If the token is the null sentinel `"#"`, return `nullptr`. Otherwise, instantiate a new node with the token value, then recursively deserialize its left subtree, then its right subtree.
- **Why Sentinels are Necessary:** A standard preorder traversal sequence alone (e.g., `[1, 2, 3]`) is ambiguous. It could represent multiple binary tree structures. Including null sentinels makes the traversal sequence unique, enabling exact reconstruction.

### Common Pitfalls
- **Skipping Null Sentinels:** Believing you can deserialize a tree without serializing its null children. Without null markers, reconstruction is impossible unless you restrict the tree shape (e.g., perfect binary trees only).


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <string>
#include <sstream>
#include <vector>
using namespace std;

struct TreeNode {
    int value;
    TreeNode *left, *right;
    TreeNode(int val) : value(val), left(nullptr), right(nullptr) {}
};

// Recursive preorder serialization helper
void serializeHelper(TreeNode* root, stringstream& ss) {
    if (root == nullptr) {
        ss << "#, "; // '#' represents a null pointer
        return;
    }
    ss << root->value << ", ";
    serializeHelper(root->left, ss);
    serializeHelper(root->right, ss);
}

string serialize(TreeNode* root) {
    stringstream ss;
    serializeHelper(root, ss);
    return ss.str();
}

// Recursive preorder deserialization helper
TreeNode* deserializeHelper(stringstream& ss) {
    string token;
    if (!getline(ss, token, ',')) return nullptr;
    
    // Trim whitespace
    while (!token.empty() && token.front() == ' ') token.erase(token.begin());
    while (!token.empty() && token.back() == ' ') token.pop_back();
    
    if (token == "#" || token.empty()) {
        return nullptr;
    }
    
    TreeNode* node = new TreeNode(stoi(token));
    node->left = deserializeHelper(ss);
    node->right = deserializeHelper(ss);
    return node;
}

TreeNode* deserialize(string data) {
    stringstream ss(data);
    return deserializeHelper(ss);
}

void printInOrder(TreeNode* root) {
    if (root) {
        printInOrder(root->left);
        cout << root->value << " ";
        printInOrder(root->right);
    }
}

void cleanTree(TreeNode *root) {
    if (root) {
        cleanTree(root->left);
        cleanTree(root->right);
        delete root;
    }
}

int main() {
    // Construct tree:
    //       1
    //      / \
    //     2   3
    //        /
    //       4
    TreeNode* root = new TreeNode(1);
    root->left = new TreeNode(2);
    root->right = new TreeNode(3);
    root->right->left = new TreeNode(4);

    string serializedStr = serialize(root);
    cout << "Serialized String:   " << serializedStr << "\n";
    // Expected: 1, 2, #, #, 3, 4, #, #, #, 

    TreeNode* reconstructed = deserialize(serializedStr);
    cout << "Reconstructed (Inorder): ";
    printInOrder(reconstructed);
    cout << "\n"; // Expected: 2 1 4 3

    cleanTree(root);
    cleanTree(reconstructed);
    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 7. Time and Space Complexity Analysis — Stack vs. Queue Bounds

### The Why
Both DFS and BFS traverse tree nodes, achieving identical $O(n)$ time complexity. However, their **space complexity** bounds differ dramatically depending on the tree's shape. Understanding these bounds is critical for choosing the right traversal to minimize memory usage.

### Core Mechanism
- **DFS (Depth-First Search):**
  - Space complexity is determined by the **height** of the tree ($h$): $O(h)$ auxiliary memory, matching the maximum number of recursive call frames (or explicit stack elements) stored simultaneously.
  - **Balanced Binary Tree:** Height $h \approx \log_2 n$. Space complexity is highly optimal: $O(\log n)$.
  - **Skewed Binary Tree (Degenerate):** Height $h \approx n$. Space complexity degrades to $O(n)$.
- **BFS (Breadth-First Search):**
  - Space complexity is determined by the maximum **width** of the tree ($w$): $O(w)$ auxiliary memory, matching the maximum number of level nodes stored in the queue simultaneously.
  - **Balanced Binary Tree:** The bottom level contains $2^h \approx n / 2$ nodes. Queue space complexity is $O(n)$.
  - **Skewed Binary Tree (Degenerate):** The tree is a single linear chain. Width is 1. Queue space complexity is $O(1)$.

### Complexity Comparison Summary

| Tree Shape | DFS Time | DFS Space (Stack) | BFS Time | BFS Space (Queue) |
|---|---|---|---|---|
| **Balanced** | $O(n)$ | $O(\log n)$ (Optimal) | $O(n)$ | $O(n)$ (Heavy) |
| **Skewed / Degenerate** | $O(n)$ | $O(n)$ (Heavy) | $O(n)$ | $O(1)$ (Optimal) |

### Common Pitfalls
- **Defaulting to BFS:** Using BFS for search problems on massive, balanced trees. The queue will store up to half the tree's nodes at once, consuming significant memory. For balanced trees, DFS is far more space-efficient.


## 8. Decision Guide — DFS vs. BFS for Tree Problems

### The Why
Choosing between DFS and BFS depends on the problem's requirements and constraints. This decision guide summarizes typical signals.

### Decision Matrix
- **Choose DFS when:**
  - The problem asks to verify paths or search for specific leaf targets (e.g., path sums, generating all root-to-leaf paths).
  - The problem requires bottom-up divide-and-conquer logic where parents need values from subtrees (e.g., subtree height, balance checks, tree diameter, LCA).
  - The tree is balanced, and you want to minimize memory footprint ($O(\log n)$ space for DFS vs. $O(n)$ for BFS).
- **Choose BFS when:**
  - The problem requires level-by-level statistics or groupings (e.g., level averages, right side view, zigzag level printing).
  - The problem asks for the shortest path from root to leaf (e.g., minimum depth of binary tree). BFS will terminate immediately upon finding the first leaf, avoiding scanning deeper branches.
  - The tree is extremely skewed/narrow, and you want to avoid recursion stack overflow ($O(1)$ space for BFS vs. $O(n)$ for DFS).


## 9. Phase Checkpoint, Cheat Sheet, and Answer Key

### Checkpoint Questions
1. Explain the primary distinction between Top-Down and Bottom-Up DFS recursion flows.
2. Why must we store the queue size (`levelSize = q.size()`) *before* starting the level loop in BFS?
3. In iterative preorder, why do we push the right child before the left child onto the stack?
4. Why does LCA in a general binary tree require postorder divide-and-conquer instead of simple value comparisons?
5. Why is a null sentinel character (like `#`) required when serializing a binary tree?
6. For a perfect binary tree of $n$ nodes, which traversal (DFS or BFS) uses less memory? Why?

### Answer Key
1. Top-down DFS passes state variables *down* through function arguments, processing calculations before recursing. Bottom-up DFS gathers values from subtrees *first*, combines them, and bubbles the result *up* to the parent.
2. If we don't save the queue size, the loop condition `i < q.size()` will dynamically evaluate new children enqueued during the loop, mixing different levels together.
3. Stack is LIFO. Pushing the right child first guarantees the left child sits on top, meaning it will be popped and processed first, maintaining preorder (left before right) traversal.
4. General binary trees lack sorted order, so we cannot make binary decisions. We must scan both subtrees bottom-up and bubble up node pointers to locate the split point where targets reside in different branches.
5. Without null sentinels, a single traversal string (like preorder) is ambiguous and can represent multiple different binary tree shapes. Sentinels uniquely define the tree structure.
6. DFS uses less memory. For a perfect binary tree, DFS space is $O(h) = O(\log n)$, while BFS queue space is $O(w) = O(n)$ because the widest level holds half the tree's nodes.

### Mini Project: Subtree Check
Implement a function `isSubtree(TreeNode* s, TreeNode* t)` to check if tree `t` is a subtree of tree `s`.
- **Strategy:** Traverse tree `s` using DFS. At each node in `s`, if its value matches `t`'s root, call a helper function `isIdentical` to check if the subtrees are structurally identical. This combines traversal and structural comparison.


In [ ]:
%%writefile temp.cpp
#include <iostream>
using namespace std;

struct TreeNode {
    int value;
    TreeNode *left, *right;
    TreeNode(int val) : value(val), left(nullptr), right(nullptr) {}
};

// Helper: Checks if two trees are structurally identical
bool isIdentical(TreeNode* s, TreeNode* t) {
    if (s == nullptr && t == nullptr) return true;
    if (s == nullptr || t == nullptr) return false;
    
    return (s->value == t->value) && 
           isIdentical(s->left, t->left) && 
           isIdentical(s->right, t->right);
}

// Main function: Traverses s and checks for subtree t
bool isSubtree(TreeNode* s, TreeNode* t) {
    if (s == nullptr) return false; // If s is empty, it cannot contain t
    if (t == nullptr) return true;  // An empty tree is always a subtree
    
    if (isIdentical(s, t)) return true;
    
    return isSubtree(s->left, t) || isSubtree(s->right, t);
}

void cleanTree(TreeNode *root) {
    if (root) {
        cleanTree(root->left);
        cleanTree(root->right);
        delete root;
    }
}

int main() {
    // Tree s:
    //       3
    //      / \
    //     4   5
    //    / \
    //   1   2
    TreeNode* s = new TreeNode(3);
    s->left = new TreeNode(4);
    s->right = new TreeNode(5);
    s->left->left = new TreeNode(1);
    s->left->right = new TreeNode(2);

    // Tree t:
    //     4
    //    / \
    //   1   2
    TreeNode* t = new TreeNode(4);
    t->left = new TreeNode(1);
    t->right = new TreeNode(2);

    cout << "Is t a subtree of s? " << (isSubtree(s, t) ? "Yes" : "No") << "\n"; // Expected: Yes

    // Modify t to make it different
    t->right->value = 9;
    cout << "After modification, is t a subtree? " << (isSubtree(s, t) ? "Yes" : "No") << "\n"; // Expected: No

    cleanTree(s);
    cleanTree(t);
    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 10. LeetCode Practice Problems

Use these problems to build fluency with DFS, BFS, and backtracking patterns.

### Top-Down & Bottom-Up DFS Patterns

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 112 | [Path Sum](https://leetcode.com/problems/path-sum/) | Easy | Section 1, directly — top-down state reduction |
| 543 | [Diameter of Binary Tree](https://leetcode.com/problems/diameter-of-binary-tree/) | Easy | Section 1, directly — bottom-up height gathering while updating diameter |
| 111 | [Minimum Depth of Binary Tree](https://leetcode.com/problems/minimum-depth-of-binary-tree/) | Easy | Can solve with DFS (divide & conquer) or BFS (shortest path — returns early) |
| 124 | [Binary Tree Maximum Path Sum](https://leetcode.com/problems/binary-tree-maximum-path-sum/) | Hard | Advanced bottom-up DFS — each node computes maximum single branch sum, while updating global path maximum |

### Tree Backtracking

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 113 | [Path Sum II](https://leetcode.com/problems/path-sum-ii/) | Medium | Section 2, directly — track path by reference and pop-back to backtrack |
| 257 | [Binary Tree Paths](https://leetcode.com/problems/binary-tree-paths/) | Easy | Traverse from root, appending node values to path string. Save when leaf is reached |
| 129 | [Sum Root to Leaf Numbers](https://leetcode.com/problems/sum-root-to-leaf-numbers/) | Medium | Pass current running number down: `currentNumber * 10 + root->value`. Sum leaf numbers |

### Tree BFS & Level Size Tracking

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 199 | [Binary Tree Right Side View](https://leetcode.com/problems/binary-tree-right-side-view/) | Medium | Section 3, directly — record last node in level size loop |
| 103 | [Binary Tree Zigzag Level Order Traversal](https://leetcode.com/problems/binary-tree-zigzag-level-order-traversal/) | Medium | Section 3, directly — alternate level insertion index direction |
| 637 | [Average of Levels in Binary Tree](https://leetcode.com/problems/average-of-levels-in-binary-tree/) | Easy | Level size loop — sum values at current level and divide by count |
| 515 | [Find Largest Value in Each Tree Row](https://leetcode.com/problems/find-largest-value-in-each-tree-row/) | Medium | Level size loop — maintain running max per level |

### Iterative & Advanced Patterns

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 236 | [Lowest Common Ancestor of a Binary Tree](https://leetcode.com/problems/lowest-common-ancestor-of-a-binary-tree/) | Medium | Section 5, general tree LCA — bubble matching nodes bottom-up |
| 297 | [Serialize and Deserialize Binary Tree](https://leetcode.com/problems/serialize-and-deserialize-binary-tree/) | Hard | Section 6, preorder DFS — represent null pointers with sentinels |
| 572 | [Subtree of Another Tree](https://leetcode.com/problems/subtree-of-another-tree/) | Easy | Section 9's Mini Project — check isIdentical on nodes during DFS walk |

### Self-Check Before Moving to `04_Self_Balancing_and_Tries`
If you can explain why top-down DFS differs from bottom-up DFS, implement level size tracking in BFS, and serialize a tree using sentinels, you have completed the core goals of this notebook. In `04_Self_Balancing_and_Tries`, we will see how self-balancing algorithms (AVL Trees) maintain logarithmic height to prevent tree degeneracy, and how Tries optimize string prefix searches.